# GenAI Pipeline — Subpillar Testing Notebook

LLM-based subpillar labelling of grants on alternative proteins, split by production platform
(Fermentation / Plant-Based / Cross-cutting — Cultivated has no subpillar and is not classified
here). Uses Claude with prompt caching via the Anthropic Python SDK. Structured like
`grant_rescat_testing.ipynb` (per-pillar prompts and datasets, selected via `AP_PILLAR`) rather
than `grant_endproduct_testing.ipynb`, since each pillar's subpillar taxonomy is completely
different, not just a different category list. See `6_subpillar/{F,PB,CC}_v1/NOTE.txt` for how
each pillar's prompt was designed.


### 1. Imports and Configuration


In [1]:
import pandas as pd
import json
import random
import time
import anthropic
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import create_model

load_dotenv()

OUTPUT_DIR = Path(".")
RAW_SUBS_DIR = Path("6_subpillar/raw_subsets/")


### 2. Data Inspection


In [2]:
EXCEL_PATH = Path("1_deduplication/raw_data/Funding2026_inscope.xlsx")
df_raw = pd.read_excel(EXCEL_PATH)

has_abstract = df_raw["Abstract"].notna() & (df_raw["Abstract"].str.strip() != "")
df = df_raw[has_abstract].reset_index(drop=True)

# Normalise the "Plant-Based" typo variant before splitting by platform (same fix as rescat).
df["Production platform"] = df["Production platform"].replace({"Plant-Based": "Plant-based"})

# "Bf" (lowercase f) is a single data-entry typo for "BF" among Fermentation rows.
df["Sub-production pillar"] = df["Sub-production pillar"].replace({"Bf": "BF"})

print(f"Raw shape: {df_raw.shape}")
print(f"Filtered shape (has abstract): {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()


Raw shape: (1678, 81)
Filtered shape (has abstract): (924, 81)

Columns: ['Title', 'Abstract', 'Original title', 'Database', 'Total amount', 'Gov contribution', 'Currency', 'Total amount (USD)', 'Gov contribution (USD)', 'Total amount (EUR)', 'Gov & NP contribution (EUR)', 'Funding decision', 'copy to external database', 'URL for announcement', 'Identification code', 'Unnamed: 15', 'Unnamed: 16', 'Notes (external)', 'Notes (internal)', 'Project lead (PI)', 'PI department', 'PI organisation', 'PI organisation type', 'PI organisation country', 'PI organisation region', 'PI organisation state', 'PI organisation zip code', 'PI organisation congressional district', 'Collaborator names', 'Collaborator institutions', 'Multiple organisation recipients', 'Date request submitted', 'Year request submitted', 'Date award announced', 'Project start date', 'Duration of award (months)', 'Project status', 'Annual expenditures', 'Funder name', 'Sub-funder', 'Funding call', 'Success rate', 'Funder type',

,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035
0,Plant2Food,The new collaborative platform Plant2Food will...,NaN,airtable,200000000,200000000,DKK,28473000.0,0.0,26000000.0,...,4333333.333,4333333.333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CO2 as a sustainable raw material in our futur...,"In a new consortium, companies and university ...",NaN,airtable,100000000,100000000,DKK,27000000.0,0.0,13000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Precision Technology: Biotechnology, smart sen...",The project has three sub-goals:\n\nDeveloping...,NaN,airtable,64200000,64200000,NOK,NaN,NaN,5585400.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SEEDFOOD: Functional and palatable plant seed ...,The Foundation has awarded one of the 2021 gra...,NaN,airtable,55900000,55900000,DKK,8172831.0,0.0,7267000.0,...,1038142.857,1038142.857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,National Alternative Proteins Innovation and K...,"To secure a continued supply of safe, tasty, ...",NaN,airtable,38000000,16001352,GBP,48859450.0,18697500.0,45220000.0,...,3173601.480,3173601.480,3173601.48,3173601.48,NaN,NaN,NaN,NaN,NaN,NaN


### 3. Platform Split

Unlike end product / award purpose, subpillar has a **completely different taxonomy per
production platform** — not just a different category list — so each platform is split into its
own dataframe here, mirroring `grant_rescat_testing.ipynb`. Cultivated has no real subpillar
(219/221 rows blank; the 2 exceptions are legacy "Meat"/"Meat, Seafood" noise) and is skipped
entirely — no dataframe, no prompt, no `AP_PILLAR` option for it.

- **Fermentation**: every row should resolve to BF / PF / Mixed. 2 of 313 rows have a genuinely
  null Sub-production pillar despite that — set aside for manual review rather than force-sampled.
- **Plant-Based**: subpillar (TF) is optional — blank is the expected outcome for 93% of rows, so
  blank rows are kept and given an explicit `"None"` sentinel category rather than dropped, so the
  balanced sample includes true negatives to test the model correctly predicts *no* TF.
- **Cross-cutting**: subpillar (CellAg / All / Agnostic) is optional the same way (73% blank). 5
  rows carrying a dropped label (2 "CM for PB", 1 "Consumer", 2 "PF" — the last a likely upstream
  Production-platform mislabeling artifact, not a real cross-cutting pattern; see
  `6_subpillar/CC_v1/NOTE.txt`) are excluded entirely, since this v1 prompt doesn't attempt them.


In [3]:
df_f = df[df["Production platform"] == "Fermentation"].reset_index(drop=True)
df_pb = df[df["Production platform"] == "Plant-based"].reset_index(drop=True)
df_cc = df[df["Production platform"] == "Cross-cutting"].reset_index(drop=True)

# Fermentation: every row must resolve to BF/PF/Mixed. Set aside the rare genuinely-null rows.
has_subpillar_f = df_f["Sub-production pillar"].notna() & (df_f["Sub-production pillar"].str.strip() != "")
needs_manual_review = df_f[~has_subpillar_f].reset_index(drop=True)
df_f = df_f[has_subpillar_f].reset_index(drop=True)

# Plant-Based / Cross-cutting: subpillar is optional, so blank is a meaningful, common ground
# state to sample as a negative example — fill with an explicit "None" sentinel category rather
# than dropping, so create_balanced_sample can include some deliberately-blank rows.
# A handful of Plant-Based rows carry algae-related mislabeling noise (Algae/CellAg/BF/PF) rather
# than a real second Plant-Based subpillar scheme — excluded rather than treated as categories.
PB_EXCLUDED_LABELS = ["Algae", "CellAg", "BF", "PF"]
df_pb = df_pb[~df_pb["Sub-production pillar"].isin(PB_EXCLUDED_LABELS)].reset_index(drop=True)
df_pb["Sub-production pillar"] = df_pb["Sub-production pillar"].fillna("None")

CC_EXCLUDED_LABELS = ["CM for PB", "Consumer", "PF"]
df_cc = df_cc[~df_cc["Sub-production pillar"].isin(CC_EXCLUDED_LABELS)].reset_index(drop=True)
df_cc["Sub-production pillar"] = df_cc["Sub-production pillar"].fillna("None")

print(f"Fermentation: {df_f.shape[0]} rows ({needs_manual_review.shape[0]} set aside, null subpillar)")
print(f"Plant-Based:  {df_pb.shape[0]} rows ({PB_EXCLUDED_LABELS} excluded)")
print(f"Cross-cutting: {df_cc.shape[0]} rows ({CC_EXCLUDED_LABELS} excluded)")


Fermentation: 177 rows (2 set aside, null subpillar)
Plant-Based:  563 rows (['Algae', 'CellAg', 'BF', 'PF'] excluded)
Cross-cutting: 51 rows (['CM for PB', 'Consumer', 'PF'] excluded)


### 4. Balanced Subset Creation


In [4]:
# Each pillar's Sub-production pillar values don't overlap at all (BF/PF/Mixed vs TF/None vs
# CellAg/All/Agnostic/None), so a single unified breakdown table (like rescat's, where research
# categories DO partially overlap across pillars) would just be block-diagonal clutter — a
# separate small breakdown per pillar is clearer here.
breakdown_f = df_f["Sub-production pillar"].value_counts()
breakdown_pb = df_pb["Sub-production pillar"].value_counts()
breakdown_cc = df_cc["Sub-production pillar"].value_counts()

print("Fermentation:\n", breakdown_f)
print("\nPlant-Based:\n", breakdown_pb)
print("\nCross-cutting:\n", breakdown_cc)


Fermentation:
 Sub-production pillar
PF       94
BF       71
Mixed    12
Name: count, dtype: int64

Plant-Based:
 Sub-production pillar
None    514
TF       49
Name: count, dtype: int64

Cross-cutting:
 Sub-production pillar
None      41
CellAg     9
All        1
Name: count, dtype: int64


In [5]:
RANDOM_STATE = 3
N_PER_CATEGORY = 20
MIN_PER_COMBO = 2  # ensure at least this many examples of each distinct multi-label combination.
# Note: real multi-label combos in "Sub-production pillar" are essentially nonexistent (the only
# one dataset-wide is the excluded Cultivated "Meat, Seafood" row), so this top-up logic is
# inherited from rescat/end-product/award-purpose but effectively inert here.

def parse_categories(series):
    """Split a comma-separated 'Sub-production pillar' string into a cleaned list of labels."""
    return series.str.split(",").apply(lambda labels: [l.strip() for l in labels])

def create_balanced_sample(df, categories, n_per_category=N_PER_CATEGORY, min_per_combo=MIN_PER_COMBO, random_state=RANDOM_STATE):
    """
    Samples up to n_per_category rows per individual Sub-production pillar label.
    A multi-label row is eligible under each of its labels, and is kept only once in the combined
    sample if picked more than once. Then tops up the sample so at least min_per_combo rows of
    each distinct multi-label combination are present, to specifically test multi-label
    classification.
    """
    labels = parse_categories(df["Sub-production pillar"])

    samples = []
    for cat in categories:
        mask = labels.apply(lambda xs: cat in xs)
        subset = df[mask]
        available = len(subset)
        if available == 0:
            print(f"  Warning: '{cat}' — no rows found, skipping.")
            continue
        n = min(n_per_category, available)
        if n < n_per_category:
            print(f"  Warning: '{cat}' — requested {n_per_category} but only {available} available, taking all.")
        samples.append(subset.sample(n=n, random_state=random_state))

    combined = pd.concat(samples) if samples else df.iloc[0:0]
    combined = combined[~combined.index.duplicated(keep="first")]

    multi_mask = labels.apply(lambda xs: len(xs) > 1)
    multi_labels = labels[multi_mask]
    combos = multi_labels.apply(lambda xs: ", ".join(sorted(xs)))

    for combo in combos.unique():
        combo_idx = combos[combos == combo].index
        group = df.loc[combo_idx]
        already = group.index.isin(combined.index).sum()
        need = min_per_combo - already
        if need <= 0:
            continue
        remaining_pool = group[~group.index.isin(combined.index)]
        take = min(need, len(remaining_pool))
        if take < need:
            print(f"  Warning: combo '{combo}' — only {already + len(remaining_pool)} rows available, wanted {min_per_combo}.")
        if take > 0:
            combined = pd.concat([combined, remaining_pool.sample(n=take, random_state=random_state)])

    combined = combined[~combined.index.duplicated(keep="first")]
    return combined.sample(frac=1, random_state=random_state).reset_index(drop=True)


In [6]:
test_data_F = create_balanced_sample(df_f, breakdown_f.index.tolist())
print(f"test_data_F: {test_data_F.shape}")
# test_data_F[["Title", "Abstract", "Sub-production pillar"]]


test_data_F: (52, 81)


In [7]:
test_data_PB = create_balanced_sample(df_pb, breakdown_pb.index.tolist())
# Revert the "None" sentinel back to blank now that sampling is done, so downstream ground-truth
# comparison treats it as empty (no subpillar) rather than the literal string "None".
test_data_PB["Sub-production pillar"] = test_data_PB["Sub-production pillar"].replace({"None": pd.NA})
print(f"test_data_PB: {test_data_PB.shape}")
# test_data_PB[["Title", "Abstract", "Sub-production pillar"]]


test_data_PB: (40, 81)


In [8]:
test_data_CC = create_balanced_sample(df_cc, breakdown_cc.index.tolist())
test_data_CC["Sub-production pillar"] = test_data_CC["Sub-production pillar"].replace({"None": pd.NA})
print(f"test_data_CC: {test_data_CC.shape}")
# test_data_CC[["Title", "Abstract", "Sub-production pillar"]]


test_data_CC: (30, 81)


### 5. Save Subsets to Excel
Allows manual check of files selected. Consider whether those in the test sets are borderline cases or clear cut.


In [9]:
################################################################################################
# PLEASE CHANGE FILENAME TO THE RANDOM SEED USED IN create_balanced_sample() FOR REPRODUCIBILITY
################################################################################################
def save_subset(df, filename, output_dir=RAW_SUBS_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / filename
    df.to_excel(path, index=False)
    print(f"Saved {len(df)} records to {path}")

save_subset(test_data_F, f"subpillar_test_data_F_rand{RANDOM_STATE}.xlsx")
save_subset(test_data_PB, f"subpillar_test_data_PB_rand{RANDOM_STATE}.xlsx")
save_subset(test_data_CC, f"subpillar_test_data_CC_rand{RANDOM_STATE}.xlsx")


Saved 52 records to 6_subpillar\raw_subsets\subpillar_test_data_F_rand3.xlsx
Saved 40 records to 6_subpillar\raw_subsets\subpillar_test_data_PB_rand3.xlsx
Saved 30 records to 6_subpillar\raw_subsets\subpillar_test_data_CC_rand3.xlsx


### 6. Load Prompt and Select Dataset


In [ ]:
AP_PILLAR = "CC"  # ← CHANGE THIS: "F", "PB", "CC" (no "CM" — Cultivated has no subpillar)

DATASETS = {
    "F": test_data_F,
    "PB": test_data_PB,
    "CC": test_data_CC,
}
DATASET = DATASETS[AP_PILLAR]
print(f"Pillar: {AP_PILLAR} | Dataset: {DATASET.shape[0]} records")

# ONLY USED AS REQUIRED FOR RE-RUN SPECIFIC RECORDS
#ids_to_test = [0, 1]
#DATASET = DATASET[DATASET["id"].isin(ids_to_test)]
#DATASET = incorrect_subpillar_data
#DATASET = manual_test_data


Pillar: PB | Dataset: 40 records


In [85]:
PROMPT_VERSION = "v1"  # ← CHANGE THIS to switch prompt version
PROMPT_PATH = f"6_subpillar/{AP_PILLAR}_{PROMPT_VERSION}/prompt_subpillar_grants_{AP_PILLAR}_{PROMPT_VERSION}.md"

def load_prompt(path=PROMPT_PATH):
    with open(path, "r", encoding="utf-8") as f:
        prompt_text = f.read()
    return prompt_text.strip()

system_prompt = load_prompt()
print(system_prompt)


You are an expert in alternative proteins and food technology.

This grant has already been classified as belonging to the Plant-based pillar. Your task is to determine whether it also uses traditional fermentation as a processing technique applied to plant proteins, based on its title and abstract.

Most Plant-based grants do not involve fermentation at all — flagging Traditional fermentation True is the exception, not the default. Only flag it True when fermentation is explicitly described as a step used to modify, process, or transform a plant protein raw material.

Key decision rule:
- A grant that applies classic microbial fermentation techniques — lactic acid fermentation, solid-state fungal or mould fermentation, koji-style or tempeh-style fermentation, kefir or yoghurt-style starter cultures — directly to a plant protein raw material (legumes, pulses, cereals, oilseed or other plant-protein side streams) in order to remove anti-nutritional factors, improve digestibility, textur

### 7. API Call with Prompt Caching


In [86]:
# API config

# Anthropic model options — pricing as of 2026-06-10.
# Verify at https://www.anthropic.com/pricing if costs may have changed.
# Model                  Input $/1M   Output $/1M   Context
# claude-haiku-4-5         $1.00         $5.00       200K
# claude-sonnet-4-6        $3.00        $15.00       1M
# claude-opus-4-8          $5.00        $25.00       1M
MODELS = {
    "haiku":  "claude-haiku-4-5",
    "sonnet": "claude-sonnet-4-6",
    "opus":   "claude-opus-4-8",
}
MODEL = MODELS["sonnet"]  # ← change this to switch model

MAX_TOKENS = 512         # max tokens in response; adjust based on expected reasoning length and cost tolerance
TEMPERATURE = 0.0        # 0.0 = deterministic; raise to ~0.3 to sample variance across REPETITIONS
CALL_DELAY = 1.0         # seconds between API calls
REQUEST_TIMEOUT = 120    # seconds before giving up on a single API call
MAX_RETRIES = 6          # retry attempts on rate-limit / transient errors
RETRY_BASE_SECONDS = 5.0  # exponential backoff base
RETRY_MAX_SECONDS = 90.0  # cap on backoff sleep

REPETITIONS = 1  # number of full runs; increase to measure output variance across runs

# ================================================================
# CHECKPOINT CONFIG
# Saves progress after each record; a run interrupted mid-way can
# be resumed without re-processing completed records. A dedicated
# subfolder keeps these runs from colliding with other stages' checkpoints.
# Set RESUME_INCOMPLETE = False to always start from scratch.
# ================================================================
CHECKPOINT_DIR = Path("checkpoints/subpillar")
RESUME_INCOMPLETE = True


In [87]:
# ================================================================
# REASONING TOGGLE
# Keep True during testing — reasoning shows WHY the model decides
# as it does, which is essential for evaluating prompt quality.
# Set to False for production runs once the prompt is validated,
# to reduce token usage.
# ================================================================
INCLUDE_REASONING = False

# Each pillar's subpillar taxonomy is completely different (not just a different category list),
# unlike rescat where every pillar's categories are drawn from the same kind of list.
# "F" uses the short codes "BF"/"PF" directly (matching the ground truth's own vocabulary)
# rather than full names — see 6_subpillar/F_v2/NOTE.txt. A third "Unclear" boolean was tried and
# reverted: it fired on 13/52 rows in the real run (far more than the ~1/52 expected), so accepting
# occasional wrong Mixed guesses is the better tradeoff than that much manual-review overhead.
PILLAR_CATS = {
    "F": ["BF", "PF"],
    "PB": ["Traditional fermentation"],
    "CC": ["CellAg", "All", "Agnostic"],
}

import re

def make_schema(cats, include_reasoning):
    """
    Builds a schema with one boolean field per category (true/false, multi-label) instead of a
    single primary/secondary pick. field_map translates the sanitised Python-safe field names
    (e.g. "Biomass_fermentation") back to the original category label (e.g. "Biomass fermentation").
    """
    def field_name(cat):
        return re.sub(r"\W+", "_", cat).strip("_")

    field_map = {field_name(cat): cat for cat in cats}
    fields = {fname: (bool, ...) for fname in field_map}
    if include_reasoning:
        fields["reasoning"] = (str, ...)
    Model = create_model("ClassificationSchema", **fields)
    return Model, field_map

ClassificationSchema, CATEGORY_FIELD_MAP = make_schema(PILLAR_CATS[AP_PILLAR], INCLUDE_REASONING)
print(f"Schema built for {AP_PILLAR}: {PILLAR_CATS[AP_PILLAR]}")

client = anthropic.Anthropic(api_key=os.getenv("CLAUDE_API_KEY"))

def classify_publication(title, abstract, system_prompt):
    user_message = f"Title: {title}\n\nAbstract: {abstract}"
    response = client.messages.parse(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        timeout=REQUEST_TIMEOUT,
        system=[
            {
                "type": "text",
                "text": system_prompt,
                "cache_control": {"type": "ephemeral"}
            }
        ],
        messages=[
            {"role": "user", "content": user_message}
        ],
        output_format=ClassificationSchema,
    )
    return response.parsed_output


Schema built for PB: ['Traditional fermentation']


### 8. Error Handling with Retry


In [88]:
def is_retryable_error(exc: Exception) -> bool:
    markers = ["503", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "429",
               "TIMEOUT", "TIMED OUT", "READTIMEOUT", "CONNECTTIMEOUT"]
    return any(m in str(exc).upper() for m in markers)

def retry_sleep_seconds(attempt: int) -> float:
    sleep = min(RETRY_MAX_SECONDS, RETRY_BASE_SECONDS * (2 ** attempt))
    jitter = random.uniform(0.0, min(3.0, sleep * 0.2))
    return sleep + jitter


In [89]:
def classify_with_error_handling(row, system_prompt):
    record_id = row["id"]
    last_error = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            result = classify_publication(row["title"], row["abstract"], system_prompt)
            if result is None:
                print(f"  Parse failed for {record_id}: model returned no structured output")
                return {"id": record_id, "status": "parse_error", "error": "no structured output"}
            output = {f"{k}_LLM": v for k, v in result.model_dump().items()} # rename columns / keys to indicate LLM output
            output["id"] = record_id
            output["status"] = "ok"
            return output
        except anthropic.APIError as e:
            last_error = e
            if attempt >= MAX_RETRIES:
                break
            if is_retryable_error(e):
                sleep_s = retry_sleep_seconds(attempt)
                print(f"  Retryable error (attempt {attempt + 1}/{MAX_RETRIES}): {e}. Sleeping {sleep_s:.1f}s.")
                time.sleep(sleep_s)
            else:
                break
    print(f"  API error for {record_id}: {last_error}")
    return {"id": record_id, "status": "api_error", "error": str(last_error)}


### 9. Checkpoint Helpers


In [90]:
def get_checkpoint_path(run_idx: int) -> Path:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    return CHECKPOINT_DIR / f"run_{run_idx}_checkpoint.json"

def save_checkpoint(run_idx: int, completed_results: list) -> None:
    path = get_checkpoint_path(run_idx)
    payload = {
        "run_idx": run_idx,
        "completed_ids": [r["id"] for r in completed_results],
        "results": completed_results,
    }
    path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

def load_checkpoint(run_idx: int):
    path = get_checkpoint_path(run_idx)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def delete_checkpoint(run_idx: int) -> None:
    path = get_checkpoint_path(run_idx)
    if path.exists():
        path.unlink()


### 10. Run on Test Data


In [91]:
# Standardise DATASET column names once, separately from the LLM-calling loop below, so this
# (and the comparison cell) can be re-run for free without re-hitting the API — e.g. after a
# kernel restart, or when re-analysing results already in memory / a checkpoint file. Grants data
# has no reliable unique id column, so use row position.
DATASET = DATASET.reset_index(drop=True).rename(columns={"Title": "title", "Abstract": "abstract"})
DATASET["id"] = DATASET.index


In [92]:
all_results = []

for rep in range(REPETITIONS):
    run_idx = rep + 1
    print(f"\n{'='*50}\nRun {run_idx} / {REPETITIONS}\n{'='*50}")

    checkpoint = load_checkpoint(run_idx) if RESUME_INCOMPLETE else None
    if checkpoint:
        completed_results = checkpoint["results"]
        completed_ids = set(checkpoint["completed_ids"])
        print(f"  Resuming: {len(completed_ids)} records already processed.")
    else:
        completed_results, completed_ids = [], set()

    remaining = DATASET[~DATASET["id"].isin(completed_ids)]
    total = len(DATASET)

    for _, row in remaining.iterrows():
        n_done = len(completed_results)
        print(f"  [{n_done + 1}/{total}] {row['id']}")
        result = classify_with_error_handling(row, system_prompt)
        result["run"] = run_idx
        completed_results.append(result)
        save_checkpoint(run_idx, completed_results)
        if n_done + 1 < total:
            time.sleep(CALL_DELAY)

    delete_checkpoint(run_idx)
    all_results.extend(completed_results)



Run 1 / 1
  [1/40] 0
  [2/40] 1
  [3/40] 2
  [4/40] 3
  [5/40] 4
  [6/40] 5
  [7/40] 6
  [8/40] 7
  API error for 7: Error code: 529 - {'type': 'error', 'error': {'type': 'overloaded_error', 'message': 'Overloaded'}, 'request_id': 'req_011CdWvKgjPzjpwLdk7VHcQt'}
  [9/40] 8
  [10/40] 9
  [11/40] 10
  [12/40] 11
  [13/40] 12
  [14/40] 13
  [15/40] 14
  [16/40] 15
  [17/40] 16
  [18/40] 17
  [19/40] 18
  [20/40] 19
  [21/40] 20
  [22/40] 21
  [23/40] 22
  [24/40] 23
  [25/40] 24
  [26/40] 25
  [27/40] 26
  [28/40] 27
  [29/40] 28
  [30/40] 29
  [31/40] 30
  [32/40] 31
  [33/40] 32
  [34/40] 33
  [35/40] 34
  [36/40] 35
  [37/40] 36
  [38/40] 37
  [39/40] 38
  [40/40] 39


In [93]:
results_df = pd.DataFrame(all_results)
print(f"\nCompleted: {len(results_df)} records across {REPETITIONS} run(s)")
print(f"Successful: {(results_df['status'] == 'ok').sum()}")
print(f"Errors: {(results_df['status'] != 'ok').sum()}")
results_df



Completed: 40 records across 1 run(s)
Successful: 39
Errors: 1


,Traditional_fermentation_LLM,id,status,run,error
0,True,0,ok,1,NaN
1,False,1,ok,1,NaN
2,True,2,ok,1,NaN
3,True,3,ok,1,NaN
4,False,4,ok,1,NaN
5,False,5,ok,1,NaN
6,True,6,ok,1,NaN
7,NaN,7,api_error,1,"Error code: 529 - {'type': 'error', 'error': {..."
8,False,8,ok,1,NaN
9,False,9,ok,1,NaN


In [94]:
# Each pillar collapses its booleans into the ground truth's single "Sub-production pillar"
# string differently — unlike rescat/end-product/award-purpose, where one build_predictions
# shape works for every pillar/category-list. A small per-pillar dispatch replaces that here.
def collapse_fermentation(row):
    true_cats = [orig for field, orig in CATEGORY_FIELD_MAP.items() if row.get(f"{field}_LLM") == True]
    bf = "BF" in true_cats
    pf = "PF" in true_cats
    if bf and pf:
        return "Mixed"
    if bf:
        return "BF"
    if pf:
        return "PF"
    return ""  # shouldn't happen if the prompt's Mixed-fallback rule is followed — worth flagging in eval, not silently handling

def collapse_plantbased(row):
    true_cats = [orig for field, orig in CATEGORY_FIELD_MAP.items() if row.get(f"{field}_LLM") == True]
    return "TF" if "Traditional fermentation" in true_cats else ""

def collapse_crosscutting(row):
    true_cats = [orig for field, orig in CATEGORY_FIELD_MAP.items() if row.get(f"{field}_LLM") == True]
    return "; ".join(true_cats)

COLLAPSE = {"F": collapse_fermentation, "PB": collapse_plantbased, "CC": collapse_crosscutting}

def build_predictions(row):
    true_cats = [orig for field, orig in CATEGORY_FIELD_MAP.items() if row.get(f"{field}_LLM") == True]
    return pd.Series({
        "raw_predicted_categories": "; ".join(true_cats),
        "predicted_subpillar": COLLAPSE[AP_PILLAR](row),
    })

results_df[["raw_predicted_categories", "predicted_subpillar"]] = results_df.apply(build_predictions, axis=1)

result_cols = ["id", "run", "raw_predicted_categories", "predicted_subpillar", "status"]
if INCLUDE_REASONING:
    result_cols.append("reasoning_LLM")

comparison = DATASET[["id", "title", "abstract", "Sub-production pillar"]].merge(
    results_df[result_cols], on="id", how="left"
)
comparison = comparison.rename(columns={"Sub-production pillar": "subpillar"})

# Ground truth here is a single string (BF/PF/Mixed/TF/CellAg/etc, or blank) with essentially no
# real multi-label combos, unlike the other Funding notebooks — but the same set-based comparison
# still works correctly: a bare string with no comma becomes a 1-element set, blank becomes an
# empty set, so exact-match/precision/recall/Jaccard all behave sensibly without modification.
def to_label_set(s, sep=","):
    if not isinstance(s, str) or not s.strip():
        return set()
    return {l.strip() for l in s.split(sep)}

comparison["subpillar_set"] = comparison["subpillar"].apply(lambda s: to_label_set(s, sep=","))
comparison["predicted_subpillar_set"] = comparison["predicted_subpillar"].apply(lambda s: to_label_set(s, sep=";"))
comparison["exact_match"] = comparison["subpillar_set"] == comparison["predicted_subpillar_set"]

def label_prf(row):
    truth, pred = row["subpillar_set"], row["predicted_subpillar_set"]
    if not truth and not pred:
        return pd.Series({"row_precision": 1.0, "row_recall": 1.0, "row_jaccard": 1.0})
    tp = len(truth & pred)
    fp = len(pred - truth)
    fn = len(truth - pred)
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    jaccard = tp / len(truth | pred) if (truth | pred) else float("nan")
    return pd.Series({"row_precision": precision, "row_recall": recall, "row_jaccard": jaccard})

comparison[["row_precision", "row_recall", "row_jaccard"]] = comparison.apply(label_prf, axis=1)

n = len(comparison)
print(f"== Subpillar ({AP_PILLAR}) ==")
print(f"Exact match accuracy:  {comparison['exact_match'].mean():.0%}  (n={n})")
print(f"Mean row precision:    {comparison['row_precision'].mean():.0%}")
print(f"Mean row recall:       {comparison['row_recall'].mean():.0%}")
print(f"Mean row Jaccard:      {comparison['row_jaccard'].mean():.0%}")

# Per-category precision/recall across the multi-label predictions
def label_series(s, sep=","):
    return s.dropna().str.split(sep).explode().str.strip().dropna()

all_cats = sorted(set(label_series(comparison["subpillar"], sep=",")) | set(label_series(comparison["predicted_subpillar"], sep=";")))
cat_rows = []
for cat in all_cats:
    truth_has = comparison["subpillar_set"].apply(lambda s: cat in s)
    pred_has  = comparison["predicted_subpillar_set"].apply(lambda s: cat in s)
    tp = int((truth_has & pred_has).sum())
    fn = int((truth_has & ~pred_has).sum())
    fp = int((~truth_has & pred_has).sum())
    n_true = int(truth_has.sum())
    recall = tp / n_true if n_true else float("nan")
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    cat_rows.append({
        "subpillar": cat, "n_true": n_true, "tp": tp, "fp": fp, "fn": fn,
        "recall": recall, "precision": precision,
    })
cat_stats = pd.DataFrame(cat_rows).set_index("subpillar")
cat_stats["recall"] = cat_stats["recall"].map(lambda x: f"{x:.0%}" if pd.notna(x) else "-")
cat_stats["precision"] = cat_stats["precision"].map(lambda x: f"{x:.0%}" if pd.notna(x) else "-")
display(cat_stats)

# Detail table
display_cols = ["id", "title", "abstract", "subpillar", "raw_predicted_categories", "predicted_subpillar"]
if INCLUDE_REASONING:
    display_cols.append("reasoning_LLM")
display_cols += ["exact_match", "row_precision", "row_recall", "row_jaccard"]
comparison[display_cols]


== Subpillar (PB) ==
Exact match accuracy:  80%  (n=40)
Mean row precision:    97%
Mean row recall:       82%
Mean row Jaccard:      80%


,n_true,tp,fp,fn,recall,precision
subpillar,,,,,,
,0,0,0,0,-,-
TF,20,13,1,7,65%,93%


,id,title,abstract,subpillar,raw_predicted_categories,predicted_subpillar,exact_match,row_precision,row_recall,row_jaccard
0,0,FAIRification of multiOmics data to link datab...,A transition towards sustainable food systems ...,TF,Traditional fermentation,TF,True,1.0,1.0,1.0
1,1,Integration of acorn flour into the food chain...,Priority challenges:\nThis project directly su...,NaN,,,True,1.0,1.0,1.0
2,2,Vegan skyr from Scandinavian ingredients every...,Purpose and goal: \n Every year 160 000 tons ...,NaN,Traditional fermentation,TF,False,0.0,NaN,0.0
3,3,Plant fermentates for better shelf life and fo...,The project's aim is to increase the use of fe...,TF,Traditional fermentation,TF,True,1.0,1.0,1.0
4,4,The impact of processing steps on the techno-f...,Reducing meat consumption and shifting to a mo...,NaN,,,True,1.0,1.0,1.0
5,5,OBTAINING AND FUNCTIONALIZATION OF LUPIN PROTE...,This project aims to obtain lupin protein isol...,NaN,,,True,1.0,1.0,1.0
6,6,Collaborative project: Development of an innov...,"The KiEFer project aims to develop new, attrac...",TF,Traditional fermentation,TF,True,1.0,1.0,1.0
7,7,Enhanced application of plant proteins and sid...,The use of plant proteins and by-products for ...,TF,,,False,NaN,0.0,0.0
8,8,MuD Proteine leguminosenfreier Anbaualternativ...,"egenstand des Projektes Plantein ist es, landw...",NaN,,,True,1.0,1.0,1.0
9,9,Joint project: Upcycling of side streams with ...,No abstract,TF,,,False,NaN,0.0,0.0


### 11. Save to Excel for Prompt Debugging
To assess how well the prompt does at getting the LLM to assign subpillar labels, save the
comparison data, then manually review what went wrong and adjust the prompt. None of this will
make it into the final workflow.

Order of working:
1. Create a new version folder for this pillar in `6_subpillar` (e.g. `F_v2`).
2. Copy in the previous prompt for that pillar. Label it with the new version number. Make updates as required based on step 6.
3. Edit Step 11 output directory (this step) and Step 6 prompt selection and input data.
4. Run the script from steps 6-11.
5. Manually review the results — both metrics and individual rows.
6. Write a text document (`NOTE.txt`) about the results and what changes you want to make to the prompt. Repeat from step 1.

Repeat for each of the 3 pillars (`AP_PILLAR = "F"`, `"PB"`, `"CC"`) — each iterates independently.


In [95]:
save_dir = Path(f"6_subpillar/{AP_PILLAR}_{PROMPT_VERSION}")
save_dir.mkdir(parents=True, exist_ok=True)

summary_df = pd.DataFrame([
    {"metric": "subpillar_exact_match_accuracy", "value": f"{comparison['exact_match'].mean():.0%}", "n": n},
    {"metric": "subpillar_mean_precision", "value": f"{comparison['row_precision'].mean():.0%}", "n": n},
    {"metric": "subpillar_mean_recall", "value": f"{comparison['row_recall'].mean():.0%}", "n": n},
    {"metric": "subpillar_mean_jaccard", "value": f"{comparison['row_jaccard'].mean():.0%}", "n": n},
])

out_path = save_dir / f"{AP_PILLAR}_{PROMPT_VERSION}_{MODEL}_results.xlsx"
with pd.ExcelWriter(out_path) as writer:
    comparison[display_cols].to_excel(writer, sheet_name="results",     index=False)
    cat_stats.to_excel(             writer, sheet_name="by_category")
    summary_df.to_excel(            writer, sheet_name="summary",       index=False)

print(f"Saved to {out_path}")


Saved to 6_subpillar\PB_v1\PB_v1_claude-sonnet-4-6_results.xlsx


In [ ]:
# Records where the LLM's predicted subpillar did not exactly match ground truth — for re-run
# with a modified prompt
incorrect_ids = comparison.loc[~comparison["exact_match"], "id"]
incorrect_subpillar_data = DATASET[DATASET["id"].isin(incorrect_ids)].reset_index(drop=True)
incorrect_subpillar_data


In [34]:
# Manually select specific rows by id for quick re-testing (paste ids from the
# comparison/results tables above). Note: the Section 10 prep cell always resets
# "id" to a fresh 0..n-1 range when it runs, so once this subset goes through the
# script again its ids won't match the ones you selected here — use "title" or
# "subpillar" to cross-reference back to the original run if needed.
manual_ids = [5,12,39]  # <- CHANGE THIS to the ids you want to re-test
manual_test_data = DATASET[DATASET["id"].isin(manual_ids)].reset_index(drop=True)
print(f"Selected {len(manual_test_data)} of {len(manual_ids)} requested ids")
manual_test_data


Selected 3 of 3 requested ids


,title,abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,2027,2028,2029,2030,2031,2032,2033,2034,2035,id
0,GoCreate,So delighted to receive confirmation from the ...,NaN,airtable,50000,50000,GBP,61261.0,61261.0,59500.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
1,MYCOFACT: The dual role of sugar transporters ...,Filamentous fungi are the main plant biomass c...,Filamentous fungi are the main plant biomass c...,Bruna national 2025,9997196,9997196,DKK,NaN,NaN,1299635.48,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12
2,Synthetically engineered microalgae for improv...,Microalgae are small microscope plants that ha...,Synthetically engineered microalgae for improv...,Dimensions,1976496,1976496,GBP,2525078.0,NaN,2352030.24,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,39
